# Urban graphs: connect spatial layers

**Atlas source reading copy · browser-local**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Urbanity/blob/a9d2b1e0d0af91dc2782f7f1d77d9bbb10c67d0f/book/notebooks/urban-graphs.ipynb)



Standard-library synthetic network, context, imagery-label and graph exercises; these executable cells do not import Urbanity or download OSM/provider data. Native Urbanity, compiled GIS, imagery and graph-ML examples in Markdown are continuations requiring their own environment and data permissions. Owner-authorized original material has no inferred public repository license.

**Provenance:** `book/notebooks/urban-graphs.ipynb` at `a9d2b1e0d0af91dc2782f7f1d77d9bbb10c67d0f`; original SHA-256 `70af78d47658709e9336b8cccf084af07fbfa83717715044f3ad4ab1888fa3af`. Original owner-authored companion incorporated at repository owner request; Urbanity source repository has no public license grant. Package Urbanity and third-party data retain separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# Urban graphs: connect spatial layers

A street-only graph misses relationships between roads, buildings, and plots. A heterogeneous urban graph keeps different kinds of spatial objects as distinct node types, then relates them with typed edges. The following is a tiny invented graph, not real city data.

![Study area to graph workflow](../_static/urbanity-pipeline.svg)

[Launch this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Urbanity/lab/lab/index.html?path=notebooks/urban-graphs.ipynb) · [Official urban-graph tutorial](https://urbanity.readthedocs.io/en/latest/tutorials/urban-graphs/)

## Try it: name the layers and relationships

Nodes in different layers can share local identifiers, so represent identity as a `(layer, id)` pair. This avoids treating building `0` and plot `0` as the same object.

In [1]:
layers = {
    "plot": ["P1", "P2"],
    "building": ["B1", "B2", "B3"],
    "street": ["S1", "S2"],
    "intersection": ["I1", "I2", "I3"],
}
relations = {
    "building_in_plot": [("B1", "P1"), ("B2", "P1"), ("B3", "P2")],
    "building_near_street": [("B1", "S1"), ("B2", "S1"), ("B3", "S2")],
    "street_meets_intersection": [("S1", "I1"), ("S1", "I2"), ("S2", "I2"), ("S2", "I3")],
}

print("Objects by layer:")
for layer, objects in layers.items():
    print(f"  {layer:12} {len(objects)}")
print("Typed relationships:")
for relation, pairs in relations.items():
    print(f"  {relation:26} {len(pairs)}")

Objects by layer:
  plot         2
  building     3
  street       2
  intersection 3
Typed relationships:
  building_in_plot           3
  building_near_street       3
  street_meets_intersection  4


In [2]:
def check_relation(source_layer, target_layer, pairs):
    """Report links whose endpoints are absent from their declared layers."""
    source_ids = set(layers[source_layer])
    target_ids = set(layers[target_layer])
    return [(source, target) for source, target in pairs
            if source not in source_ids or target not in target_ids]

checks = {
    "building_in_plot": ("building", "plot"),
    "building_near_street": ("building", "street"),
    "street_meets_intersection": ("street", "intersection"),
}
for relation, (source, target) in checks.items():
    invalid = check_relation(source, target, relations[relation])
    print(relation, "valid" if not invalid else f"invalid links: {invalid}")

building_in_plot valid
building_near_street valid
street_meets_intersection valid


Typed links encode different ideas: containment (`building_in_plot`), proximity (`building_near_street`), and connectivity (`street_meets_intersection`). The chosen rule matters: nearest-neighbour links, shared boundaries, and geometric intersection are not interchangeable.

## Native Urbanity workflow

The official Urbanity `UrbanGraph` workflow creates the spatial layers, initializes selected relationships, and can prepare graph-learning representations. Run this in the supported native environment, not this browser kernel.

```python
from urbanity import Map

city = Map(country="Singapore")
city.add_polygon_boundary("path/to/study-area.geojson")
urban_graph = city.get_urban_graph(bandwidth=100)
urban_graph.initialize_edges(
    building_neighbours="knn", knn=3, distance=100
)
for layer, frame in urban_graph.geo_store.items():
    print(layer, len(frame))
```

See [Urban Graphs](https://urbanity.readthedocs.io/en/latest/tutorials/urban-graphs/) for the current node/edge types, construction choices, visualisations, and save/load workflow.

## Reflect

- Which relationships are directional, and which might be symmetric?
- How could an incorrect nearest-neighbour distance change the graph?
- What would you validate before using these links in a scientific analysis?